In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# An Interpreter for a Simple Programming Language

In this notebook we develop an interpreter for a small programming language.

In [ ]:
#r "LanguageSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "LanguageSyntax/bin/Debug/net9.0/LanguageSyntax.dll"
open LanguageSyntax

The typed AST distinguishes numeric literals, variable names, expressions and statements. The shared `LanguageSyntax` project also supports the extensions used in the exercise notebook.

We allow both *single-line comments* and *multi-line comments*.
- The regular expression `/\*(.|\n)*?\*/` recognizes multi-line comments.
  Multi-line comments start with the string `/*` and end with the string `*/`.
  Note the use of the *non-greedy* quantor `*?`.  If we have code like
  ```
  /* blah */ a := 1; /* blub */
  ```
  the greedy quantor would recognize the whole line as one comment. 
- The regular expression `//.*` recognizes single-line comments.
  A single line comment starts with the string `//` and extends to the end of the line.

FsLex specification:

```fsharp
{
open LanguageSyntax
open LanguageSyntax.Parser
open FSharp.Text.Lexing
let lexeme (buffer: LexBuffer<char>) = System.String(buffer.Lexeme)
let number (text: string) =
    let normalized = text.Replace("_", "")
    if normalized.IndexOfAny([|'.'; 'e'; 'E'|]) >= 0 then Real(System.Double.Parse(normalized, System.Globalization.CultureInfo.InvariantCulture))
    else Integer(System.Numerics.BigInteger.Parse(normalized, System.Globalization.CultureInfo.InvariantCulture))
}
let digit = ['0'-'9']
let digits = digit (digit | '_')*
let letter = ['a'-'z' 'A'-'Z']
rule token = parse
 | [' ' '\t' '\r']+ { token lexbuf }
 | '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
 | "//" [^'\n']* { token lexbuf }
 | "/*" { comment lexbuf; token lexbuf }
 | digits ('.' digit+)? (['e' 'E'] ['+' '-']? digit+)? { NUMBER(number (lexeme lexbuf)) }
 | '"' ([^'"' '\\' '\n'] | '\\' [^'\n'])* '"' { STRING(System.Text.Json.JsonSerializer.Deserialize<string>(lexeme lexbuf)) }
 | letter (letter | digit | '_')* {
     match lexeme lexbuf with
     | "if" -> IF | "while" -> WHILE | "for" -> FOR | "function" -> FUNCTION | "return" -> RETURN
     | name -> IDENTIFIER name }
 | ":=" { ASSIGN } | "==" { EQ } | "!=" { NE } | "<=" { LE } | ">=" { GE }
 | "&&" { AND } | "||" { OR }
 | '+' { PLUS } | '-' { MINUS } | '*' { TIMES } | '/' { DIVIDE } | '%' { MODULO }
 | '<' { LT } | '>' { GT } | '!' { NOT }
 | '(' { LPAREN } | ')' { RPAREN } | '{' { LBRACE } | '}' { RBRACE }
 | ';' { SEMICOLON } | ',' { COMMA }
 | eof { EOF }
 | _ { failwithf "Illegal character at line %d, column %d" lexbuf.StartPos.Line (lexbuf.StartPos.Column + 1) }
and comment = parse
 | "*/" { () }
 | '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; comment lexbuf }
 | eof { failwith "Unterminated block comment" }
 | _ { comment lexbuf }

```

The token `NUMBER` carries an exact `Integer` or a floating-point `Real` value. Integer literals use `BigInteger`, so their size is limited only by available memory.

The *keywords* `'if'` and `'while'` have to be dealt with separately as they are syntactical identical to identifiers. The map `Keywords` shown below maps every keyword to its token type.

When an identifier is read, we first have to check whether the identifier is one of our *keywords*.  If so, we assign the corresponding token type that is stored in the map `Keywords`.  Otherwise, the token type is set to `IDENTIFIER`.

FsYacc declares a token case for each operator and delimiter.

The *white space* characters *blank*, *tabulator*, and *carriage return* are ignored.

Syntactically, *newline* characters are ignored. However, we still need to keep track of them in order to know the current line number, which is used for error messages.

The lex buffer tracks line and column positions. `Parse.tokens` exposes them alongside the token values.

The lexer rejects unknown characters, including their line and column, rather than silently discarding them.

In [ ]:
let testScanner file =
    let source = System.IO.File.ReadAllText file
    printfn "%s" source
    printfn "%A" (Parse.tokens source)

In [ ]:
testScanner "sum.sl"

In [ ]:
testScanner "factorial.sl"

Below is the grammar for a simple `C`-like language:
```
program
    : 𝜆 
    | stmnt program
    
stmnt 
    : IF '(' bool_expr ')' stmnt                 
    | WHILE '(' bool_expr ')' stmnt
    | '{' program '}' 
    | IDENTIFIER ':=' expr ';'  
    | expr ';'       

bool_expr 
    : expr '==' expr     
    | expr '!=' expr     
    | expr '<=' expr     
    | expr '>=' expr     
    | expr '<'  expr      
    | expr '>'  expr     
 
expr: expr '+' product                 
    | expr '-' product
    | product
              
product
    : product '*' factor               
    | product '/' factor
    | product '%' factor 
    | factor

factor
    : '(' expr ')' 
    | NUMBER
    | IDENTIFIER
    | IDENTIFIER '(' expr_list ')'

expr_list
    : 𝜆 
    | ne_expr_list

ne_expr_list
    : expr
    | expr ',' ne_expr_list
```

The start production is `program`, which consumes all input and EOF.

FsYacc specification (the simple evaluator below uses the basic statement forms):

```fsharp
%{
open LanguageSyntax
%}
%token <LanguageSyntax.Value> NUMBER
%token <string> IDENTIFIER STRING
%token IF WHILE FOR FUNCTION RETURN
%token PLUS MINUS TIMES DIVIDE MODULO ASSIGN EQ NE LT GT LE GE AND OR NOT
%token LPAREN RPAREN LBRACE RBRACE SEMICOLON COMMA EOF
%start program
%type <LanguageSyntax.Statement> program
%%
program: statements EOF { Block $1 }
statements: { [] } | nonemptyStatements { $1 }
nonemptyStatements: statement { [$1] } | statement nonemptyStatements { $1 :: $2 }
statement: IF LPAREN expr RPAREN statement { If($3, $5) }
         | WHILE LPAREN expr RPAREN statement { While($3, $5) }
         | FOR LPAREN assign SEMICOLON expr SEMICOLON assign RPAREN statement { For($3, $5, $7, $9) }
         | FUNCTION IDENTIFIER LPAREN identifiers RPAREN statement { Define($2, $4, $6) }
         | RETURN expr SEMICOLON { Return $2 }
         | LBRACE statements RBRACE { Block $2 }
         | assign SEMICOLON { $1 }
         | expr SEMICOLON { Expression $1 }
assign: IDENTIFIER ASSIGN expr { Assign($1, $3) }
identifiers: { [] } | identifierList { $1 }
identifierList: IDENTIFIER { [$1] } | identifierList COMMA IDENTIFIER { $1 @ [$3] }
expr: expr OR conjunction { Binary(Or, $1, $3) } | conjunction { $1 }
conjunction: conjunction AND comparison { Binary(And, $1, $3) } | comparison { $1 }
comparison: sum EQ sum { Binary(Equal, $1, $3) }
          | sum NE sum { Binary(NotEqual, $1, $3) }
          | sum LT sum { Binary(Less, $1, $3) }
          | sum GT sum { Binary(Greater, $1, $3) }
          | sum LE sum { Binary(LessEqual, $1, $3) }
          | sum GE sum { Binary(GreaterEqual, $1, $3) }
          | sum { $1 }
sum: sum PLUS product { Binary(Add, $1, $3) }
   | sum MINUS product { Binary(Subtract, $1, $3) }
   | product { $1 }
product: product TIMES factor { Binary(Multiply, $1, $3) }
       | product DIVIDE factor { Binary(Divide, $1, $3) }
       | product MODULO factor { Binary(Modulo, $1, $3) }
       | factor { $1 }
factor: LPAREN expr RPAREN { $2 }
      | NUMBER { Literal $1 }
      | STRING { Literal(Text $1) }
      | IDENTIFIER { Variable $1 }
      | IDENTIFIER LPAREN expressions RPAREN { Call($1, $3) }
      | NOT factor { Not $2 }
      | MINUS factor { Negative $2 }
expressions: { [] } | expressionList { $1 }
expressionList: expr { [$1] } | expressionList COMMA expr { $1 @ [$3] }

```

An example program that conforms to this grammar is stored in the file `sum.sl`.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "sum.sl")

A program is a list of statements.

The generated parser returns a typed `Statement` tree; a conversion creates the Graphviz representation.

In [ ]:
#load "../Support/Visualization.fsx"
open Visualization

let opName = function
    | Add -> "+" | Subtract -> "-" | Multiply -> "*" | Divide -> "/" | Modulo -> "%"
    | Equal -> "==" | NotEqual -> "!=" | Less -> "<" | Greater -> ">" | LessEqual -> "<=" | GreaterEqual -> ">="
    | And -> "&&" | Or -> "||"
let valueText = function
    | Integer n -> string n
    | Real n -> n.ToString("G17", System.Globalization.CultureInfo.InvariantCulture)
    | Text s -> s
    | Boolean b -> string b
let rec exprTree = function
    | Literal value -> Leaf(valueText value)
    | Variable name -> Leaf name
    | Binary(op, left, right) -> Node(opName op, [exprTree left; exprTree right])
    | Not expr -> Node("!", [exprTree expr])
    | Negative expr -> Node("-", [exprTree expr])
    | Call(name, args) -> Node("call " + name, List.map exprTree args)
let rec statementTree = function
    | Block statements -> Node("block", List.map statementTree statements)
    | Assign(name, expr) -> Node(":=", [Leaf name; exprTree expr])
    | Expression expr -> Node("expression", [exprTree expr])
    | If(test, body) -> Node("if", [exprTree test; statementTree body])
    | While(test, body) -> Node("while", [exprTree test; statementTree body])
    | For(init, test, update, body) -> Node("for", [statementTree init; exprTree test; statementTree update; statementTree body])
    | Define(name, parameters, body) -> Node("function " + name, [Node("parameters", List.map Leaf parameters); statementTree body])
    | Return expr -> Node("return", [exprTree expr])
let parse file =
    let ast = Parse.file file
    printfn "%s" (treeToDot (statementTree ast))
    ast

`parse file` reads the program, creates a fresh lex buffer and prints its DOT tree.

In [ ]:
parse "sum.sl" |> ignore

F# infers the mutually recursive types from their `let rec` group; forward declarations are unnecessary.

Executing a block folds over its statements. Each statement receives the current variable map and returns the next map.

`execute io values statement` implements assignment, expression evaluation, conditionals and loops. The explicit IO record makes examples reproducible while preserving an interactive `main file` entry point.

`evaluateBool` evaluates a comparison from the same variable environment.

`evaluate` handles numeric literals, variables, arithmetic and the `read`/`print` built-ins. The variable map and IO record retain `Value`: integer addition, subtraction, multiplication and modulo stay exact, while `/` produces a `Real`, as in the original Python interpreter. `read` parses an integer and `print` formats it without conversion to `float`.

### Numeric values and comparisons

The exercise interpreter uses the same `Value` domain. These shared helpers compare integers directly and compare a mixed integer/real pair against the real value actually represented in memory. The floor quotient gives modulo the sign of its divisor, including negative operands. The `Text` and `Boolean` comparison cases are used by the exercise extensions.

In [ ]:
open System.Numerics

let number = function
    | (Integer _ | Real _) as value -> value
    | value -> failwithf "Expected a number, got %A" value
let asNumber = function
    | Integer n -> float n
    | Real n -> n
    | value -> failwithf "Expected a number, got %A" value

// Compare the integer with the actual represented real, without rounding it first.
let compareIntegerReal (integer: BigInteger) (real: float) =
    if System.Double.IsNaN real then failwith "NaN cannot be ordered"
    elif System.Double.IsPositiveInfinity real then -1
    elif System.Double.IsNegativeInfinity real then 1
    else
        let whole = BigInteger real
        let result = compare integer whole
        if result <> 0 then result else compare (float integer) real
let compareValues left right =
    match left, right with
    | Integer a, Integer b -> compare a b
    | Integer a, Real b -> compareIntegerReal a b
    | Real a, Integer b -> -(compareIntegerReal b a)
    | Real a, Real b -> compare a b
    | Text a, Text b -> System.String.CompareOrdinal(a, b)
    | Boolean a, Boolean b -> compare a b
    | _ -> failwithf "Cannot compare %A and %A" left right
let equalValues left right =
    match left, right with
    | Real a, Real b -> a = b
    | Integer _, Real b when System.Double.IsNaN b -> false
    | Real a, Integer _ when System.Double.IsNaN a -> false
    | (Integer _ | Real _), (Integer _ | Real _) -> compareValues left right = 0
    | _ -> left = right

// .NET integer division truncates; Python's integer modulo uses a floor quotient.
let floorDivide (left: BigInteger) (right: BigInteger) =
    if right.IsZero then failwith "Division by zero"
    let quotient, remainder = left / right, left % right
    if not remainder.IsZero && left.Sign <> right.Sign then quotient - BigInteger.One else quotient

// Python modulo has the divisor's sign, including an exact zero remainder.
let floatModulo (left: float) (right: float) =
    if right = 0.0 then failwith "Modulo by zero"
    let remainder = left % right
    if remainder = 0.0 then System.Math.CopySign(0.0, right)
    elif (remainder < 0.0) <> (right < 0.0) then remainder + right
    else remainder


Division returns a floating-point value even for two integers. To preserve Python semantics for large operands, `divideIntegers` rounds the exact ratio to the nearest binary double, with ties going to the even significand. This also handles ratios of integers that individually exceed the floating-point range. Mixed real arithmetic uses the ordinary .NET floating-point operations. Real modulo starts with the native remainder, then applies Python’s divisor-sign rule; this avoids forming an overflowing quotient and preserves signed zero.

In [ ]:
// Round the exact ratio once, rather than rounding each large operand to float.
let divideIntegers (numerator: BigInteger) (denominator: BigInteger) =
    if denominator.IsZero then failwith "Division by zero"
    let sign = if (numerator.Sign < 0) <> (denominator.Sign < 0) then -1.0 else 1.0
    let a, b = BigInteger.Abs numerator, BigInteger.Abs denominator
    if a.IsZero then sign * 0.0
    else
        let candidate = a.GetBitLength() - b.GetBitLength()
        if candidate > 1024L then failwith "Integer division result is too large for a real value"
        elif candidate < -1075L then sign * 0.0
        else
            let power = int candidate
            let belowPower =
                if power >= 0 then a < (b <<< power)
                else (a <<< -power) < b
            let exponent = if belowPower then power - 1 else power
            // Normal doubles have 53 significant bits; subnormals use units of 2^-1074.
            let scale = if exponent < -1022 then 1074 else 52 - exponent
            let scaledA, scaledB =
                if scale >= 0 then a <<< scale, b
                else a, b <<< -scale
            let quotient, remainder = scaledA / scaledB, scaledA % scaledB
            let twiceRemainder = remainder * 2I
            let rounded =
                if twiceRemainder > scaledB || (twiceRemainder = scaledB && not quotient.IsEven)
                then quotient + 1I else quotient
            let result = System.Math.ScaleB(float rounded, -scale)
            if System.Double.IsInfinity result then failwith "Integer division result is too large for a real value"
            sign * result

let arithmetic op left right =
    match left, right with
    | Integer a, Integer b ->
        match op with
        | Add -> Integer(a + b) | Subtract -> Integer(a - b) | Multiply -> Integer(a * b)
        | Divide -> Real(divideIntegers a b)
        | Modulo -> Integer(a - floorDivide a b * b)
        | _ -> failwith "Expected an arithmetic operator"
    | _ ->
        let a, b = asNumber left, asNumber right
        match op with
        | Add -> Real(a + b) | Subtract -> Real(a - b) | Multiply -> Real(a * b)
        | Divide -> Real(divide a b) | Modulo -> Real(floatModulo a b)
        | _ -> failwith "Expected an arithmetic operator"


In [ ]:
type IO = { Read: unit -> Value; Write: Value -> unit }

let rec evaluate io values = function
    | Literal value -> number value
    | Variable name ->
        match Map.tryFind name values with
        | Some value -> value
        | None -> failwithf "Undefined variable: %s" name
    | Call("read", []) -> number (io.Read())
    | Call("print", [expr]) -> io.Write(evaluate io values expr); Integer 0I
    | Negative expr ->
        match evaluate io values expr with
        | Integer n -> Integer(-n) | Real n -> Real(-n)
        | _ -> failwith "Unary minus expects a number"
    | Binary(op, left, right) ->
        arithmetic op (evaluate io values left) (evaluate io values right)
    | _ -> failwith "Expression is outside the simple language"

let evaluateBool io values = function
    | Binary(op, left, right) ->
        let l, r = evaluate io values left, evaluate io values right
        match op with
        | Equal -> equalValues l r | NotEqual -> not (equalValues l r)
        | Less -> compareValues l r < 0 | Greater -> compareValues l r > 0
        | LessEqual -> compareValues l r <= 0 | GreaterEqual -> compareValues l r >= 0
        | _ -> failwith "Expected a comparison"
    | _ -> failwith "Expected a comparison"

let rec execute io values = function
    | Block statements -> List.fold (execute io) values statements
    | Assign(name, expr) -> Map.add name (evaluate io values expr) values
    | Expression expr -> evaluate io values expr |> ignore; values
    | If(test, body) -> if evaluateBool io values test then execute io values body else values
    | While(test, body) ->
        let rec repeat current =
            if evaluateBool io current test then repeat (execute io current body) else current
        repeat values
    | _ -> failwith "Statement is outside the simple language"

let run io source = execute io Map.empty (Parse.program source)
let main file =
    let read () =
        printf "Please enter a natural number: "
        Integer(BigInteger.Parse(System.Console.ReadLine(), System.Globalization.CultureInfo.InvariantCulture))
    let io = { Read = read; Write = (valueText >> printfn "%s") }
    run io (System.IO.File.ReadAllText file)


In [ ]:
printfn "%s" (System.IO.File.ReadAllText "sum.sl")

In [ ]:
let output = ResizeArray<Value>()
let demoIO = { Read = (fun () -> Integer 10I); Write = output.Add }
let values = run demoIO (System.IO.File.ReadAllText "sum.sl")
checkNotebook (List.ofSeq output = [Integer 5050I])
printfn "%A" values

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "factorial.sl")

In [ ]:
output.Clear()
run demoIO (System.IO.File.ReadAllText "factorial.sl") |> ignore
checkNotebook (List.ofSeq output = [Integer 3628800I])
printfn "%A" output

### Exact integers and Python arithmetic

The checks below would fail if the interpreter rounded every literal or input to `float`.

In [ ]:
output.Clear()
let exactInput = { demoIO with Read = (fun () -> Integer 9007199254740993I) }
let exact = run exactInput "x := read(); y := x + 1; print(y);"
checkNotebook (exact["x"] = Integer 9007199254740993I)
checkNotebook (List.ofSeq output = [Integer 9007199254740994I])
let numeric = run demoIO "q := 3 / 2; r := -7 % 3; s := 7 % -3; rounded := 9007199254740993 / 3;"
checkNotebook (numeric["q"] = Real 1.5)
checkNotebook (numeric["r"] = Integer 2I && numeric["s"] = Integer -2I)
checkNotebook (numeric["rounded"] = Real 3002399751580331.0)
let compared = run demoIO "x := 0; if (9007199254740993 > 9007199254740992.0) { x := 1; }"
checkNotebook (compared["x"] = Integer 1I)
printfn "%s" (valueText exact["y"])


### Rendered program tree

The diagram makes the nesting and operator structure visible in Jupyter.

In [ ]:
let svg = statementTree (Parse.file "sum.sl") |> treeToDot |> renderDot
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(svg), "text/html") |> ignore
